# 3D arm — CT nodule malignancy (Kaggle GPU)

Runs the **3D ResNet-10** experiments with the identical protocol used for the
2D and 2.5D arms: same HU window, same splits, same label policy, same loss,
optimiser, augmentation, early stopping and metrics. Only the input
representation and the encoder change.

**Before running**
1. *Settings → Accelerator →* **GPU T4 x2** (or P100)
2. *Settings → Internet →* **On** (needed to clone the repo)
3. *Data → Add Input →* attach **both** datasets:
   `ct-nodule-volumes` (volumes + binary package) and
   `ct-nodule-metadata` (3-class split CSVs)


## 1. GPU check

In [ ]:
!nvidia-smi
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 2. Locate the attached dataset

In [ ]:
import os, glob
for root in sorted(glob.glob("/kaggle/input/*")):
    print(root)
    for f in sorted(os.listdir(root))[:10]:
        p = os.path.join(root, f)
        sz = os.path.getsize(p)/1e9 if os.path.isfile(p) else None
        print("   ", f, f"({sz:.2f} GB)" if sz else "(folder)")

Paths are **discovered**, not hard-coded: Kaggle has more than one input layout
(`/kaggle/input/<slug>` on older sessions, `/kaggle/input/datasets/<user>/<slug>`
on newer ones), so the cell below searches for the files themselves.

The 3-class CSVs come from the **metadata** dataset: two files were truncated to
0 bytes when the volumes dataset was staged, so that copy is not trustworthy.
`qc_exclusions.csv` is used as the marker for the good copy because it is the
file that was empty there. Every CSV is asserted non-empty before training.


In [ ]:
import os, glob

def find_one(name, where="/kaggle/input"):
    """Locate a file anywhere under the input mounts, ignoring 0-byte copies."""
    hits = [p for p in glob.glob(f"{where}/**/{name}", recursive=True)
            if os.path.isfile(p) and os.path.getsize(p) > 0]
    if not hits:
        raise FileNotFoundError(
            f"no non-empty {name} under {where}; attached inputs: "
            f"{sorted(os.listdir(where))}")
    return sorted(hits, key=len)[0]

PACK = find_one("volumes_fp16.npy")
META = os.path.dirname(find_one("qc_exclusions.csv"))      # the clean 3-class CSVs
BIN  = os.path.dirname(find_one("binary_train.csv"))
print("volume pack :", PACK)
print("metadata dir:", META)
print("binary dir  :", BIN)

# config.py wants CT_DATA_ROOT/metadata/*.csv and /kaggle/input is read-only,
# so build a writable root of symlinks.
ROOT3C = "/kaggle/working/data_3class"
os.makedirs(f"{ROOT3C}/metadata", exist_ok=True)
for f in sorted(os.listdir(META)):
    if not f.endswith(".csv"):
        continue
    dst = f"{ROOT3C}/metadata/{f}"
    if os.path.islink(dst) or os.path.exists(dst):
        os.unlink(dst)                       # drop stale links from earlier attempts
    os.symlink(f"{META}/{f}", dst)

os.environ["CT_DATA_ROOT"]   = ROOT3C
os.environ["CT_BINARY_ROOT"] = BIN
os.environ["CT_VOLUME_PACK"] = PACK

# fail loudly rather than training on a silently empty split
for d in (f"{ROOT3C}/metadata", BIN):
    print(d)
    for f in sorted(os.listdir(d)):
        if not f.endswith(".csv"):
            continue
        n = os.path.getsize(os.path.realpath(os.path.join(d, f)))
        assert n > 0, f"EMPTY CSV: {d}/{f}"
        print(f"   ok {f:32s} {n/1e6:7.2f} MB")


## 3. Clone the code

Private repo, so this needs a GitHub token with `repo` scope. Store it as a
Kaggle Secret (*Add-ons → Secrets*, name it `GITHUB_TOKEN`) so it is never
written into the notebook.

In [ ]:
from kaggle_secrets import UserSecretsClient
TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
USER, REPO = "naazzarov", "CT-2D-2.5D-3D-DeepLearning"
!git clone -q https://{TOKEN}@github.com/{USER}/{REPO}.git /kaggle/working/repo
%cd /kaggle/working/repo
!ls src/3d

## 4. Verify the data

Asserts zero patient/nodule leakage, the agreed binary counts, and that volumes
load at the expected shape. Raises rather than warns.

In [ ]:
import sys; sys.path.insert(0, "/kaggle/working/repo")
from src.common.data import load_splits, load_ssl_pool, assert_no_leakage, verify_binary_counts
from src.common.data3d import VolumeDataset, volume_train_statistics

s3 = load_splits("3class"); sb = load_splits("binary")
assert_no_leakage(s3, load_ssl_pool()); verify_binary_counts(sb)
assert_no_leakage(sb, load_ssl_pool(), reference=s3)
print("3-class:", {k: len(v) for k, v in s3.items()})
print("binary :", {k: len(v) for k, v in sb.items()})
mean, std = volume_train_statistics(s3["train"], limit=100)
print("volume intensity mean %.4f std %.4f" % (mean, std))
x, y = VolumeDataset(s3["val"], mean, std)[0]
print("sample shape", tuple(x.shape), "label", y)

## 5. Speed check

Expect roughly 0.3–0.8 min/epoch. An Apple M2 managed 13.5.

In [ ]:
import time, torch, sys
sys.path.insert(0, "/kaggle/working/repo/src/3d")
from model import ResNet10Classifier, n_params
dev="cuda"; m=ResNet10Classifier().to(dev)
opt=torch.optim.AdamW(m.parameters(), lr=1e-3); crit=torch.nn.CrossEntropyLoss()
x=torch.randn(16,1,104,72,80,device=dev); y=torch.randint(0,3,(16,),device=dev)
for _ in range(3): opt.zero_grad(); crit(m(x),y).backward(); opt.step()
torch.cuda.synchronize(); t=time.perf_counter()
for _ in range(10): opt.zero_grad(); crit(m(x),y).backward(); opt.step()
torch.cuda.synchronize(); per=(time.perf_counter()-t)/10
print(f"{per:.3f} s/step -> {1876/16*per/60:.1f} min/epoch")
print("params %.2fM" % (n_params(m)/1e6))

## 6. Model A — 3-class supervised, no augmentation

In [ ]:
!for s in 0 1 2; do python src/3d/train_supervised.py --task 3class \
  --init random --aug none --tag model_a_noaug --seed $s \
  --epochs 80 --patience 20 --batch-size 16 --workers 2 --eval-test 2>&1 \
  | grep -vE "^ep +[0-9]+ \|"; done

## 7. Model C — 3-class supervised, mild augmentation

In [ ]:
!for s in 0 1 2; do python src/3d/train_supervised.py --task 3class \
  --init random --aug mild --tag model_c_mild --seed $s \
  --epochs 80 --patience 20 --batch-size 16 --workers 2 --eval-test 2>&1 \
  | grep -vE "^ep +[0-9]+ \|"; done

## 8. Binary sensitivity — mild augmentation

In [ ]:
!for s in 0 1 2; do python src/3d/train_supervised.py --task binary \
  --init random --aug mild --tag binary_mild --seed $s \
  --epochs 80 --patience 20 --batch-size 16 --workers 2 --eval-test 2>&1 \
  | grep -vE "^ep +[0-9]+ \|"; done

## 9. MoCo v2 pretraining

The long one: 5133 training-patient volumes, no labels. Kaggle sessions cap at
12 h, so 100 epochs is the safe starting point. The checkpoint is copied to
`/kaggle/working` immediately, which persists when you *Save Version*.

In [ ]:
!python src/3d/moco_pretrain.py --epochs 100 --batch-size 32 --workers 2
!mkdir -p /kaggle/working/ckpt && cp results/3d/moco/moco_encoder.pt /kaggle/working/ckpt/ && echo saved

## 10. Model B — MoCo fine-tuning

In [ ]:
!for s in 0 1 2; do python src/3d/train_supervised.py --task 3class \
  --init moco --moco-ckpt results/3d/moco/moco_encoder.pt --aug none \
  --tag model_b_noaug --seed $s \
  --epochs 80 --patience 20 --batch-size 16 --workers 2 --eval-test 2>&1 \
  | grep -vE "^ep +[0-9]+ \|"; done

## 11. Push results back to GitHub

In [ ]:
!cp -r results/3d /kaggle/working/results_3d_backup
!git -c user.email="b.nazarov0309@gmail.com" -c user.name="Bahodir" add -A results/3d
!git -c user.email="b.nazarov0309@gmail.com" -c user.name="Bahodir" commit -q -m "3D results from Kaggle GPU run"
!git push && echo pushed